In [67]:
using XLSX
using DataFrames

function read_excel_to_dict(file_path::AbstractString)
    # Initialize an empty dictionary to store the sheets
    excel_data = Dict{String, DataFrame}()
    
    # Open the Excel file
    xlsx_file = XLSX.readxlsx(file_path)
    
    # Iterate through each sheet in the Excel file
    for sheet in 1:length(XLSX.sheetnames(xlsx_file))
        # read the sheet name and store it for indexing
        sheet_name = XLSX.sheetnames(xlsx_file)[sheet]
        # Convert the sheet data to a DataFrame (from DataFrames.jl package)
        # Store the DataFrame in the dictionary with the sheet name as the key
        excel_data[sheet_name] = DataFrame(XLSX.readtable(file_path,XLSX.sheetnames(xlsx_file)[sheet]))
    end
    
    return excel_data
end

# Example usage:
file_path = "Model_data.xlsx"  # Replace with the path to your Excel file
vaccine_dict = read_excel_to_dict(file_path)

Dict{String, DataFrame} with 17 entries:
  "Production Capacity"       => 21×2 DataFrame…
  "Penta  Pricing"            => 20×21 DataFrame…
  "DTP Pricing"               => 10×21 DataFrame…
  "Td pricing"                => 10×26 DataFrame…
  "OPV Pricing"               => 13×21 DataFrame…
  "HPV Pricing"               => 13×21 DataFrame…
  "Hexa Pricing"              => 4×3 DataFrame…
  "Tetanus Pricing"           => 16×20 DataFrame…
  "country demand by bracket" => 205×33 DataFrame…
  "DT Pricing"                => 4×21 DataFrame…
  "Measles Pricing"           => 5×21 DataFrame…
  "HepB Pricing"              => 22×20 DataFrame…
  "vaccine-producer list"     => 24×19 DataFrame…
  "MMR Pricing"               => 9×21 DataFrame…
  "IPV Pricing"               => 15×16 DataFrame…
  "vaccine-antigen list"      => 24×11 DataFrame…
  "MR Pricing"                => 6×21 DataFrame…

In [ ]:
#producer list generation
producer_list = names(vaccine_dict["vaccine-producer list"])
popfirst!(producer_list)
producer_list

In [ ]:
#vaccine list
vaccine_list = vaccine_dict["vaccine-producer list"][!,"Vaccine"]

In [ ]:
#antigen list
antigen_list = names(vaccine_dict["vaccine-antigen list"])
popfirst!(antigen_list)
antigen_list

In [85]:
vaccine_demand_data = vaccine_dict["country demand by bracket"];

In [108]:
A = ["Measles","Mumps","Rubella"]
V = ["M","MR","MMR"]

A_v = Dict("M" => ["Measles"], "MR" => ["Measles","Rubella"], "MMR" => ["Measles","Mumps", "Rubella"])

V_a = Dict()
for a in A
    vector_a = []
    for v in keys(A_v)
        if a in A_v[v]
            push!(vector_a, v)
        end
    end
    V_a[a] = vector_a
end

P = ["Serum Institute of India","PT Bio Farma", "GlaxoSmithKline", "Biological E. Limited"]
P_v = Dict("M" => ["Serum Institute of India", "PT Bio Farma"],"MR" => ["Serum Institute of India", "Biological E. Limited"], "MMR" => ["Serum Institute of India","GlaxoSmithKline"])

V_p = Dict()
for p in P
    vector_p = []
    for v in keys(P_v)
        if p in P_v[v]
            push!(vector_p, v)
        end
    end
    V_p[p] = vector_p
end

tmin = 1
tmax = 10
T = [t for t in tmin:tmax]
T_initial = [t for t in tmin-1:tmax];

In [109]:
V_p

Dict{Any, Any} with 4 entries:
  "Biological E. Limited"    => Any["MR"]
  "GlaxoSmithKline"          => Any["MMR"]
  "PT Bio Farma"             => Any["M"]
  "Serum Institute of India" => Any["MMR", "MR", "M"]

In [99]:
d_real = vaccine_demand_data[((vaccine_demand_data.Vaccine .== "Measles") .| (vaccine_demand_data.Vaccine .== "Mumps") .| (vaccine_demand_data.Vaccine .== "Rubella")).& (vaccine_demand_data.wb_status .== "Low Income"), Cols(23:end)][2:4,:]

d = Dict()
for a in 1:length(A)
    for t in 1:length(T)
        d[A[a],T[t]] = d_real[a,t]
    end
end

In [100]:
d

Dict{Any, Any} with 30 entries:
  ("Measles", 2)  => 4.88313e7
  ("Measles", 7)  => 1.6719e8
  ("Mumps", 5)    => 1.76734e6
  ("Measles", 6)  => 1.28323e8
  ("Measles", 10) => 8.07117e7
  ("Rubella", 10) => 7.98573e7
  ("Mumps", 6)    => 1.72289e6
  ("Mumps", 8)    => 1.94807e6
  ("Rubella", 1)  => 2.81766e7
  ("Rubella", 4)  => 5.63706e7
  ("Mumps", 10)   => 2.0867e6
  ("Measles", 8)  => 7.70035e7
  ("Rubella", 2)  => 1.9746e7
  ("Measles", 1)  => 7.19892e7
  ("Rubella", 3)  => 3.26817e7
  ("Measles", 4)  => 1.28172e8
  ("Mumps", 1)    => 1.09537e6
  ("Rubella", 5)  => 5.16345e7
  ("Measles", 9)  => 1.25477e8
  ("Measles", 5)  => 8.37032e7
  ("Rubella", 8)  => 7.12522e7
  ("Rubella", 9)  => 1.24647e8
  ("Mumps", 4)    => 9.38541e5
  ("Mumps", 2)    => 1.62132e6
  ("Rubella", 6)  => 7.99463e7
  ⋮               => ⋮

In [110]:
#this is synthetic data until real data is produced
k_rand = [2110062340  57513030 18467290 18467290 1543537077 11006234 57513030 18467290 1543737077 1543370779; 11006234 57513030 18467290 1543737077 1543370779 11006234 57513030 18467290 1543737077 1543370779; 11006234 57513030 18467290 1543370779 1594337077 11006234 57513030 18467290 1543737077 1543370779]
k = Dict()
for v in 1:length(V)
    for t in 1:length(T)
        k[V[v],T[t]] = k_rand[v,t]
    end
end

In [111]:
k

Dict{Any, Any} with 30 entries:
  ("MMR", 1) => 11006234
  ("MMR", 2) => 57513030
  ("MMR", 8) => 18467290
  ("M", 6)   => 11006234
  ("MR", 4)  => 1543737077
  ("MR", 10) => 1543370779
  ("M", 8)   => 18467290
  ("MR", 6)  => 11006234
  ("M", 9)   => 1543737077
  ("MMR", 9) => 1543737077
  ("MR", 2)  => 57513030
  ("MMR", 6) => 11006234
  ("M", 1)   => 2110062340
  ("M", 10)  => 1543370779
  ("MR", 3)  => 18467290
  ("MR", 9)  => 1543737077
  ("MMR", 3) => 18467290
  ("MMR", 5) => 1594337077
  ("MMR", 4) => 1543370779
  ("MR", 8)  => 18467290
  ("M", 5)   => 1543537077
  ("M", 7)   => 57513030
  ("M", 4)   => 18467290
  ("M", 3)   => 18467290
  ("MR", 1)  => 11006234
  ⋮          => ⋮

In [102]:
#build price dict

In [118]:
r_rand = [0.53745 0.5776 0.53745 9999 9999; 0.5679 .82 0.53745 9999 9999; 0.59835 8 0.53745 0.024 9999;;; 0.5679 .52 0.53745 9999 9999; 0.6288 0.2864 0.53745 9999 9999; 0.2864 0.024 0.53745 9999 9999;;; 0.59835 .456 0.53745 9999 9999; 0.2864 .44 0.53745 .756 0.756; 0.32405 0.2864 0.53745 0.75 0.56;;; 0.6288 4 0.53745 0.56 9999; 10 4 0.53745 9999 9999; 15 15 0.53745 9999 9999;;; 0.32405 0.024 0.53745 9999 9999; 0.024 8 0.53745 9999 9999; 30 30 0.53745 9999 9999]
#r_rand = [5 2; 5 2; 6 6;;; 5 2; 5 2; 6 6;;; 10 4; 10 4; 12 12;;; 10 4; 10 4; 12 12;;; 20 8; 20 8; 24 24]
T = 5
r = Dict()
for v in 1:length(V)
    for p in 1:length(P)
        for t in 1:length(T)
            println("Vaccine $V[v], Producer $P[p], Time $T[t]")
            #println(r_rand[v,p,t])
            #r[V[v],P[p],T[t]] = r_rand[v,p,t]
        end
    end
end

Vaccine ["M", "MR", "MMR"][v], Producer ["Serum Institute of India", "PT Bio Farma", "GlaxoSmithKline", "Biological E. Limited"][p], Time 5[t]
Vaccine ["M", "MR", "MMR"][v], Producer ["Serum Institute of India", "PT Bio Farma", "GlaxoSmithKline", "Biological E. Limited"][p], Time 5[t]
Vaccine ["M", "MR", "MMR"][v], Producer ["Serum Institute of India", "PT Bio Farma", "GlaxoSmithKline", "Biological E. Limited"][p], Time 5[t]
Vaccine ["M", "MR", "MMR"][v], Producer ["Serum Institute of India", "PT Bio Farma", "GlaxoSmithKline", "Biological E. Limited"][p], Time 5[t]
Vaccine ["M", "MR", "MMR"][v], Producer ["Serum Institute of India", "PT Bio Farma", "GlaxoSmithKline", "Biological E. Limited"][p], Time 5[t]
Vaccine ["M", "MR", "MMR"][v], Producer ["Serum Institute of India", "PT Bio Farma", "GlaxoSmithKline", "Biological E. Limited"][p], Time 5[t]
Vaccine ["M", "MR", "MMR"][v], Producer ["Serum Institute of India", "PT Bio Farma", "GlaxoSmithKline", "Biological E. Limited"][p], Time 5[t]

In [113]:
r

Dict{Any, Any} with 12 entries:
  ("MR", "Biological E. Limited", 5)     => 9999.0
  ("M", "Serum Institute of India", 5)   => 0.53745
  ("MMR", "PT Bio Farma", 5)             => 8.0
  ("MR", "GlaxoSmithKline", 5)           => 0.53745
  ("MMR", "Biological E. Limited", 5)    => 0.024
  ("M", "PT Bio Farma", 5)               => 0.5776
  ("M", "Biological E. Limited", 5)      => 9999.0
  ("M", "GlaxoSmithKline", 5)            => 0.53745
  ("MR", "PT Bio Farma", 5)              => 0.82
  ("MMR", "GlaxoSmithKline", 5)          => 0.53745
  ("MMR", "Serum Institute of India", 5) => 0.59835
  ("MR", "Serum Institute of India", 5)  => 0.5679